# 00v2 - A0: Fix the CV measurement gate

**Plan item A0** (2026-08-25 reorientation, see
[[project-rsna-phase-status]] / the linked strategy artifact): the
current gate is pooled OOF macro-AUC over the 58 gold rows, moving
+-0.02-0.05 across seeds, while every experiment in the plan expects a
0.01-0.10 effect. This notebook builds the fixed instrument, in four
parts:

- **Part A** - a second, low-variance gauge (macro-AUC vs. held-out
  weak labels, thousands of studies) plus a `worse_of_two` selection
  rule: a candidate must not regress either gauge. Pure logic, no image
  data needed - built and validated here against real local data
  shapes (58 gold / 4,349 weak rows from `data/raw/train.csv`).
- **Part A** also adds a per-label gate: macro alone can't distinguish
  a real ~10/12-label effect from a null riding on one noisy label.
- **Part B** - checks whether the existing `GroupKFold` (grouped only
  by `report_group_key()`) also blocks *scanner* leakage. Oleksii
  Zhukov's DICOM-metadata probe on this competition measured a 0.0534
  macro-AUC drop (0.6516 -> 0.5981) going from random to
  scanner-grouped folds on metadata-only prediction - pure site
  memorization. This needs the full DICOM tree (`train_series/`),
  which only exists on Kaggle - those cells are guarded to skip
  locally and print what they need.

Sources: Reference A's own `worse_of_two` design, our own Phase-4 audit
(caught a +0.042 selection leak), Oleksii Zhukov (scanner-grouped
folds, this competition), stevenleehans (per-label noise diagnostics,
this competition). Full detail in the strategy artifact linked from
`project-rsna-phase-status` memory.

## Setup

Part A below is CSV/logic only and runs identically on Kaggle or
locally. Part B needs the full DICOM tree (`train_series/`), Kaggle
only - guarded per cell rather than at the top, so Part A still runs
end-to-end wherever this notebook is opened.

In [1]:
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

# Kaggle mounts competitions under /kaggle/input/competitions/<slug>/ (no
# plain /kaggle/input/<slug>/ for this competition, confirmed 2026-08-16).
_KAGGLE_RAW = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
ON_KAGGLE = _KAGGLE_RAW.exists()

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RAW_DIR = _KAGGLE_RAW if ON_KAGGLE else REPO_ROOT / "data" / "raw"
print("RAW_DIR:", RAW_DIR, "| ON_KAGGLE:", ON_KAGGLE)

# Copied from src/config.py::OFFICIAL_LABEL_COLUMNS / FINDINGS - Kaggle
# can't import this repo, so notebooks repeat these constants by hand.
OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL",
    "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus",
    "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA",
    "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA",
    "effusion": "Effusion",
    "synovitis": "Synovitis",
    "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion",
    "fracture": "Fracture",
}
FINDINGS = list(OFFICIAL_LABEL_COLUMNS.keys())
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

random.seed(42)
np.random.seed(42)


RAW_DIR: C:\Users\alher\Desktop\RSNA_Knee_Abnormality_Detection\data\raw | ON_KAGGLE: False


## Part A - gate instrumentation

### A.1 - metric functions (copied from `src/evaluate.py`)

Same formula already graduated and unit-tested (`tests/test_evaluate.py`)
- repeated here rather than imported, for Kaggle portability.

In [2]:
def macro_roc_auc(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> float:
    if list(y_true.columns) != list(y_pred.columns):
        raise ValueError("y_true and y_pred must have matching columns.")
    return float(np.mean([
        roc_auc_score(y_true[col], y_pred[col]) for col in y_true.columns
    ]))


def per_finding_roc_auc(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> pd.Series:
    return pd.Series({
        col: roc_auc_score(y_true[col], y_pred[col]) for col in y_true.columns
    })


### A.2 - load the real gold/weak split (local CSV, real scale)

Same null-pattern logic as `src/data.py::load_gold_labels` (58 rows,
all 12 labels populated) - repeated inline here so the rest of Part A
can demonstrate its plumbing against the real 58/4,349 split sizes,
not a toy shape.

In [3]:
train = pd.read_csv(RAW_DIR / "train.csv")
n_labels_present = train[LABEL_COLS].notna().sum(axis=1)
gold_mask = n_labels_present == len(LABEL_COLS)

gold_studies = train.loc[gold_mask, "StudyInstanceUID"]
weak_studies = train.loc[~gold_mask, "StudyInstanceUID"]

print(f"gold studies: {len(gold_studies)}")
print(f"weak studies: {len(weak_studies)}")
assert len(gold_studies) + len(weak_studies) == len(train)


gold studies: 58
weak studies: 4349


### A.3 - `worse_of_two`: a candidate must not regress either gauge

Two gauges, two different biases (stevenleehans, discussion 735304):
the 58 gold rows are hand-picked to all carry at least one positive
finding (not representative of true prevalence), while the weak-label
gauge inherits the labeler's own systematic errors instead. Neither
gauge alone is trustworthy; `worse_of_two` requires a candidate to hold
up on both.

Tolerances are **required arguments, no invented defaults** - `gold_tol`
should start around the measured 0.02-0.05 pooled-OOF-macro seed
variance (Fase 4/5), `weak_tol` needs its own measurement once a real
model produces repeated-seed OOF runs against the weak-label gauge (no
such run exists yet - stevenleehans's own 0.0020 is a different
pipeline's same-config seed-repeat, not directly this project's
number).

In [4]:
# A candidate passes only if it doesn't regress beyond noise on either
# gauge. Returns a diagnostic dict, not just a bool, so a failure can be
# attributed to the gauge that actually caught it.
def worse_of_two(baseline_gold: float, candidate_gold: float,
                  baseline_weak: float, candidate_weak: float,
                  gold_tol: float, weak_tol: float) -> dict:
    gold_delta = candidate_gold - baseline_gold
    weak_delta = candidate_weak - baseline_weak
    gold_ok = gold_delta >= -gold_tol
    weak_ok = weak_delta >= -weak_tol
    return {
        "passed": gold_ok and weak_ok,
        "gold_delta": gold_delta,
        "weak_delta": weak_delta,
        "gold_ok": gold_ok,
        "weak_ok": weak_ok,
    }


Demo with three synthetic scenarios (pure logic - no image data
needed to validate the rule itself):

In [5]:
GOLD_TOL, WEAK_TOL = 0.03, 0.01  # starting points, see A.3 note above

scenarios = {
    "improves both gauges": worse_of_two(0.571, 0.590, 0.540, 0.552, GOLD_TOL, WEAK_TOL),
    "gold up, weak regresses beyond tol": worse_of_two(0.571, 0.610, 0.540, 0.520, GOLD_TOL, WEAK_TOL),
    "both within tolerance (noise, not a real change)": worse_of_two(0.571, 0.580, 0.540, 0.535, GOLD_TOL, WEAK_TOL),
}
for name, result in scenarios.items():
    print(f"{name}: passed={result['passed']}  (gold_delta={result['gold_delta']:+.3f}, weak_delta={result['weak_delta']:+.3f})")


improves both gauges: passed=True  (gold_delta=+0.019, weak_delta=+0.012)
gold up, weak regresses beyond tol: passed=False  (gold_delta=+0.039, weak_delta=-0.020)
both within tolerance (noise, not a real change): passed=True  (gold_delta=+0.009, weak_delta=-0.005)


### A.4 - per-label gate: is this a broad effect or one noisy label riding along?

stevenleehans's encoder-scaling post (this competition): a real effect
moved ~10/12 labels; a null dressed as a win moved ~5/12 and rode on
the noisiest label. Macro alone can't tell these apart - this checks
how many findings actually moved in the direction the macro delta
claims. `tol=0.03` and `min_concordant=7` (a bare majority of 12) are
**starting heuristics, not independently derived** - same hedge the
source material gives its own ~0.03 estimate.

In [6]:
# Diagnose whether a macro-AUC change reflects a broad effect across
# findings or a narrow one riding on a single noisy label.
def per_label_gate(baseline_auc: pd.Series, candidate_auc: pd.Series,
                    tol: float = 0.03, min_concordant: int = 7) -> dict:
    delta = candidate_auc - baseline_auc
    macro_delta = float(delta.mean())
    moved = delta[delta.abs() >= tol]
    concordant = int((np.sign(moved) == np.sign(macro_delta)).sum()) if macro_delta != 0 else 0
    return {
        "macro_delta": macro_delta,
        "n_labels_moved": int(len(moved)),
        "n_concordant": concordant,
        "broad_effect": concordant >= min_concordant,
        "per_label_delta": delta,
    }


Demo: a broad true effect (most findings move together) vs. a
narrow one (macro looks the same, but it's really one label moving a
lot while the rest sit flat):

In [7]:
baseline_auc = pd.Series(0.60, index=FINDINGS)

broad_candidate = baseline_auc.copy()
broad_candidate[:] += np.random.default_rng(0).uniform(0.02, 0.06, size=len(FINDINGS))

narrow_candidate = baseline_auc.copy()
narrow_candidate["synovitis"] += 0.40  # one label carries the whole macro move
narrow_candidate[[f for f in FINDINGS if f != "synovitis"]] += 0.005  # rest ~flat

for name, candidate in [("broad true effect", broad_candidate), ("narrow / one-label-riding", narrow_candidate)]:
    result = per_label_gate(baseline_auc, candidate)
    print(f"{name}: macro_delta={result['macro_delta']:+.3f}  "
          f"n_moved={result['n_labels_moved']}/12  n_concordant={result['n_concordant']}/12  "
          f"broad_effect={result['broad_effect']}")


broad true effect: macro_delta=+0.041  n_moved=9/12  n_concordant=9/12  broad_effect=True
narrow / one-label-riding: macro_delta=+0.038  n_moved=1/12  n_concordant=1/12  broad_effect=False


### A.5 - combine into one gate decision

Applies `worse_of_two` to both macro gauges, then `per_label_gate` to
the gold per-label deltas (the 58-row set is our primary interpretable
signal; the weak per-label breakdown is noisier and treated as
secondary, not checked here).

In [8]:
def gate_decision(baseline_gold_auc: pd.Series, candidate_gold_auc: pd.Series,
                   baseline_weak_macro: float, candidate_weak_macro: float,
                   gold_tol: float, weak_tol: float,
                   label_tol: float = 0.03, min_concordant: int = 7) -> dict:
    baseline_gold_macro = float(baseline_gold_auc.mean())
    candidate_gold_macro = float(candidate_gold_auc.mean())

    macro_check = worse_of_two(baseline_gold_macro, candidate_gold_macro,
                                baseline_weak_macro, candidate_weak_macro,
                                gold_tol, weak_tol)
    label_check = per_label_gate(baseline_gold_auc, candidate_gold_auc,
                                  tol=label_tol, min_concordant=min_concordant)

    return {
        "passed": macro_check["passed"] and label_check["broad_effect"],
        "macro_check": macro_check,
        "label_check": label_check,
    }


In [9]:
demo = gate_decision(
    baseline_gold_auc=baseline_auc,
    candidate_gold_auc=broad_candidate,
    baseline_weak_macro=0.540,
    candidate_weak_macro=0.552,
    gold_tol=GOLD_TOL, weak_tol=WEAK_TOL,
)
print("overall passed:", demo["passed"])
print("macro_check:", {k: v for k, v in demo["macro_check"].items() if k != "per_label_delta"})
print("label_check broad_effect:", demo["label_check"]["broad_effect"],
      "| n_concordant:", demo["label_check"]["n_concordant"])


overall passed: True
macro_check: {'passed': True, 'gold_delta': 0.041078990611080446, 'weak_delta': 0.01200000000000001, 'gold_ok': True, 'weak_ok': True}
label_check broad_effect: True | n_concordant: 9


### Part A summary

`worse_of_two`, `per_label_gate`, and `gate_decision` are validated
against real local data shapes (58 gold / 4,349 weak) and synthetic
edge cases. Pending review, these graduate into `src/evaluate.py`
alongside `macro_roc_auc`/`per_finding_roc_auc`.

Computing a **real** `weak_macro` number needs actual OOF predictions
from a trained model scored against the weak-label gauge - that only
exists once a model is retrained under the (possibly fixed) folds from
Part B below, so it isn't fabricated here.

## Part B - does `report_group_key()` already block scanner leakage?

**Needs Kaggle** (full DICOM tree under `train_series/`). Locally this
section prints what it would do and stops - run it for real on Kaggle
and report the printed numbers back.

`report_group_key()` copied from `src/labelers.py` (small, no need to
import the rest of that module here).

In [10]:
import hashlib
import re
import unicodedata


def report_group_key(report_text) -> str:
    if not isinstance(report_text, str):
        normalized = ""
    else:
        t = unicodedata.normalize("NFKD", report_text.lower())
        t = "".join(ch for ch in t if not unicodedata.combining(ch))
        normalized = re.sub(r"\s+", " ", t).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


### B.1 - extract a per-study scanner fingerprint

One representative DICOM file per study (first series, first file,
header-only read via `stop_before_pixels=True` - no pixel decoding, so
this is cheap even across all 4,407 studies). Fingerprint = whichever
of these identifying tags are present: `Manufacturer`,
`ManufacturerModelName`, `InstitutionName`, `DeviceSerialNumber`,
`MagneticFieldStrength`, `StationName`.

In [11]:
if not ON_KAGGLE:
    print("Skipping B.1 - needs the full DICOM tree. Run on Kaggle: "
          "builds `scanner_fp` (Series -> per-study fingerprint tuple) "
          "for all studies in train_series.csv.")
else:
    import pydicom

    FINGERPRINT_TAGS = ["Manufacturer", "ManufacturerModelName", "InstitutionName",
                        "DeviceSerialNumber", "MagneticFieldStrength", "StationName"]

    train_series = pd.read_csv(RAW_DIR / "train_series.csv")
    first_series = train_series.drop_duplicates("StudyInstanceUID", keep="first")

    fingerprints = {}
    for i, row in enumerate(first_series.itertuples(index=False), start=1):
        series_dir = RAW_DIR / "train_series" / row.StudyInstanceUID / row.SeriesInstanceUID
        files = sorted(series_dir.glob("*.dcm"))
        if not files:
            fingerprints[row.StudyInstanceUID] = None
            continue
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        fingerprints[row.StudyInstanceUID] = tuple(
            str(getattr(ds, tag, None)) for tag in FINGERPRINT_TAGS
        )
        if i % 500 == 0:
            print(f"  {i}/{len(first_series)} studies fingerprinted")

    scanner_fp = pd.Series(fingerprints, name="scanner_fp")
    scanner_fp.index.name = "StudyInstanceUID"
    print("done:", len(scanner_fp), "studies |", scanner_fp.nunique(), "distinct fingerprints")


Skipping B.1 - needs the full DICOM tree. Run on Kaggle: builds `scanner_fp` (Series -> per-study fingerprint tuple) for all studies in train_series.csv.


### B.2 - build the current (report-only) folds and check for scanner leakage

Same construction as `src/data.py::load_training_labels` (GroupKFold,
grouped by `report_group_key()` only, `config.CV_FOLDS = 5`). For each
fold, a scanner fingerprint that appears in *both* that fold's train
and validation rows is a leak - the model could memorize the scanner
at train time and get credit for it at validation time.

In [12]:
if not ON_KAGGLE:
    print("Skipping B.2 - depends on B.1's scanner_fp. Run on Kaggle.")
else:
    CV_FOLDS = 5
    reports = train.set_index("StudyInstanceUID")["Report"]
    group_keys = reports.apply(report_group_key)

    gkf = GroupKFold(n_splits=CV_FOLDS)
    fold = pd.Series(-1, index=reports.index, dtype=int)
    for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=group_keys.to_numpy())):
        fold.iloc[val_idx] = fold_idx

    fp_aligned = scanner_fp.reindex(reports.index)

    leaked_fingerprints = set()
    leaked_studies = 0
    for k in range(CV_FOLDS):
        train_fps = set(fp_aligned[fold != k].dropna())
        val_fps = set(fp_aligned[fold == k].dropna())
        overlap = train_fps & val_fps
        leaked_fingerprints |= overlap
        leaked_studies += fp_aligned[(fold == k) & fp_aligned.isin(overlap)].shape[0]

    print(f"scanner fingerprints split across a fold's train/val boundary: {len(leaked_fingerprints)}")
    print(f"studies affected (in the validation side of a leaked fingerprint): {leaked_studies} / {len(reports)}")


Skipping B.2 - depends on B.1's scanner_fp. Run on Kaggle.


### B.3 - if leaking, fix by grouping on the union of report-template and scanner groups

Combining the two grouping keys naively (e.g. a tuple of both) is
wrong: if study X shares a report template with Y, and Y shares a
scanner with Z, X and Z must land in the same fold even though X and Z
share neither key directly. Union-find over "shares a report template"
OR "shares a scanner fingerprint" edges gives the correct connected
components.

In [13]:
if not ON_KAGGLE:
    print("Skipping B.3 - depends on B.1/B.2. Run on Kaggle.")
else:
    class UnionFind:
        def __init__(self, items):
            self.parent = {item: item for item in items}

        def find(self, x):
            while self.parent[x] != x:
                self.parent[x] = self.parent[self.parent[x]]
                x = self.parent[x]
            return x

        def union(self, a, b):
            ra, rb = self.find(a), self.find(b)
            if ra != rb:
                self.parent[ra] = rb

    uf = UnionFind(reports.index)

    for _, idx in group_keys.groupby(group_keys).groups.items():
        idx = list(idx)
        for other in idx[1:]:
            uf.union(idx[0], other)

    for _, idx in fp_aligned.dropna().groupby(fp_aligned.dropna()).groups.items():
        idx = list(idx)
        for other in idx[1:]:
            uf.union(idx[0], other)

    combined_group = reports.index.map(uf.find)

    fold_fixed = pd.Series(-1, index=reports.index, dtype=int)
    for fold_idx, (_, val_idx) in enumerate(gkf.split(reports, groups=combined_group)):
        fold_fixed.iloc[val_idx] = fold_idx

    # re-verify both guarantees under the fixed folds
    report_leaks = 0
    for _, idx in group_keys.groupby(group_keys).groups.items():
        if fold_fixed.loc[list(idx)].nunique() > 1:
            report_leaks += 1

    scanner_leaks = 0
    fp_fixed = scanner_fp.reindex(reports.index)
    for k in range(CV_FOLDS):
        train_fps = set(fp_fixed[fold_fixed != k].dropna())
        val_fps = set(fp_fixed[fold_fixed == k].dropna())
        scanner_leaks += len(train_fps & val_fps)

    print(f"report-template groups split across folds (fixed): {report_leaks}")
    print(f"scanner fingerprints split across folds (fixed): {scanner_leaks}")
    print("\nfold sizes:", fold_fixed.value_counts().sort_index().to_dict())
    is_gold = reports.index.isin(gold_studies)
    print("gold studies per fold:", pd.Series(is_gold, index=reports.index).groupby(fold_fixed).sum().to_dict())


Skipping B.3 - depends on B.1/B.2. Run on Kaggle.


### B.4 - fold count: 5 vs. 4

Mammography 1st place went 5->4 folds deliberately for more positives
per split. Diagnostic only, using the fixed (report+scanner) groups
from B.3 - compares gold-study count per fold, the tightest resource
here (58 total).

In [14]:
if not ON_KAGGLE:
    print("Skipping B.4 - depends on B.3's combined_group. Run on Kaggle.")
else:
    for n_folds in (4, 5):
        gkf_n = GroupKFold(n_splits=n_folds)
        fold_n = pd.Series(-1, index=reports.index, dtype=int)
        for fold_idx, (_, val_idx) in enumerate(gkf_n.split(reports, groups=combined_group)):
            fold_n.iloc[val_idx] = fold_idx
        gold_per_fold = pd.Series(is_gold, index=reports.index).groupby(fold_n).sum()
        print(f"n_folds={n_folds}: gold studies per fold = {gold_per_fold.to_dict()} "
              f"(min={gold_per_fold.min()})")


Skipping B.4 - depends on B.3's combined_group. Run on Kaggle.


## Interpretation and decision

**Real numbers from the Kaggle run (2026-08-25, full 4,407-study corpus):**

- **B.2 (report-only folds):** 51 scanner fingerprints split across a
  fold's train/val boundary, affecting **4,399 / 4,407 studies (99.8%)**
  - essentially the entire corpus. Report-only grouping does NOT block
  scanner leakage on this data; the concern this section exists to check
  was real, and worse in scope than Zhukov's own 0.0534-macro-drop probe
  implied (that number was a metadata-only prediction drop, not a
  measurement of how many studies are affected).
- **B.3 (report + scanner union-find):** 0 report-template leaks, 0
  scanner leaks - the fix works. Fold sizes came out uneven (1307 /
  775 / 775 / 775 / 775) because a few scanner fingerprints connect a
  large fraction of the corpus into one big component; this is a
  structural consequence of the corpus (most studies really do share a
  handful of scanners), not a bug in the union-find, and doesn't hurt
  the pooled-OOF macro-AUC gate itself (fold-size balance only matters
  for per-fold training economics, not for the final pooled score).
  Gold-per-fold under 5 folds: {0: 17, 1: 7, 2: 15, 3: 8, 4: 11} - fold
  1's 7 gold studies is uncomfortably thin for the pooled 58-row gate.
- **B.4 (fold count):** n_folds=4 gives gold-per-fold {17, 11, 19, 11}
  (min 11); n_folds=5 gives {17, 7, 15, 8, 11} (min 7). 4 folds is
  clearly better balanced for the tightest resource here.

**Decisions made and already graduated to `src/` (this session):**

1. `src/data.py::build_group_ids()` - the union-find over one or more
   grouping-key Series, graduated with unit tests
   (`tests/test_data.py`).
2. `src/data.py::build_scanner_fingerprints()` - the per-study DICOM
   header scan (Kaggle-only at real scale; unit-tested here against
   synthetic DICOM files via the existing `_write_dicom` helper).
3. `src/data.py::load_training_labels()` - now takes a REQUIRED
   `scanner_fingerprints` argument and groups folds on
   `build_group_ids(report_group_key, scanner_fingerprints)` instead of
   report-template alone. Required, not defaulted, so no caller can
   silently reproduce the measured 4,399/4,407-study leak.
4. `src/config.py::CV_FOLDS` - changed 5 -> 4, per the B.4 measurement.

Full test suite green (48 passed, 1 skipped) after these changes.

**Still open:** the next real training run needs to call
`build_scanner_fingerprints()` on Kaggle to get real fingerprints for
all 4,407 studies, then pass them into `load_training_labels()` - this
notebook's B.1 loop is the reference implementation for that, not yet
wired into a production training script (no such script exists yet -
A3 rebuilds preprocessing/training from scratch, per the plan).